In [0]:
# ============================================================
# JOB MARKET ANALYSIS PIPELINE
# Lever API → AWS S3 RAW
# ============================================================

import requests
import json
from datetime import datetime, timezone


# ============================================================
# 1. CONFIGURATION
# ============================================================

# AWS S3 RAW location
S3_RAW_PATH = "s3://job-market-analysis-pipeline/raw-data/lever"


# ------------------------------------------------------------
# Use ONLY the Lever companies that passed your API
# verification in Colab.
# ------------------------------------------------------------

LEVER_COMPANIES = {
    "Palantir": "palantir",
    "Spotify": "spotify",
    "Zoox": "zoox",
    "Kraken": "kraken",
    "DEUNA": "deuna"
}


# ============================================================
# 2. CURRENT INGESTION DATE
# ============================================================

ingestion_date = datetime.now(
    timezone.utc
).strftime("%Y-%m-%d")

run_timestamp = datetime.now(
    timezone.utc
).isoformat()

print(f"Ingestion date: {ingestion_date}")
print(f"Run timestamp: {run_timestamp}")


# ============================================================
# 3. LEVER API FUNCTION
# ============================================================

def fetch_lever_jobs(company_name, site):

    url = (
        f"https://api.lever.co/v0/postings/"
        f"{site}"
    )

    params = {
        "mode": "json"
    }

    print("\n------------------------------------------")
    print(f"Fetching: {company_name}")
    print(f"Endpoint: {url}")

    try:

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        print(
            f"HTTP Status: "
            f"{response.status_code}"
        )

        response.raise_for_status()

        data = response.json()

        # ----------------------------------------------------
        # Lever should return a list of job postings
        # ----------------------------------------------------

        if not isinstance(data, list):

            raise ValueError(
                f"Expected a list response for "
                f"{company_name}"
            )

        print(
            f"Jobs received: "
            f"{len(data)}"
        )

        return data


    except requests.exceptions.Timeout:

        print(
            f"ERROR: Request timeout - "
            f"{company_name}"
        )

        return None


    except requests.exceptions.RequestException as e:

        print(
            f"ERROR: API request failed - "
            f"{company_name}"
        )

        print(str(e))

        return None


    except json.JSONDecodeError:

        print(
            f"ERROR: Invalid JSON response - "
            f"{company_name}"
        )

        return None


    except Exception as e:

        print(
            f"ERROR: {company_name}"
        )

        print(str(e))

        return None


# ============================================================
# 4. INGESTION RESULT TRACKING
# ============================================================

successful_companies = []

failed_companies = []


# ============================================================
# 5. FETCH API DATA AND WRITE TO S3
# ============================================================

for company_name, site in LEVER_COMPANIES.items():

    # --------------------------------------------------------
    # Fetch API data
    # --------------------------------------------------------

    data = fetch_lever_jobs(
        company_name,
        site
    )


    # --------------------------------------------------------
    # Handle failed API request
    # --------------------------------------------------------

    if data is None:

        failed_companies.append(
            company_name
        )

        continue


    # --------------------------------------------------------
    # Build RAW payload
    #
    # We preserve the original Lever job records and
    # add ingestion metadata at the outer level.
    # --------------------------------------------------------

    output = {

        "source": "lever",

        "company": company_name,

        "site": site,

        "ingestion_timestamp": run_timestamp,

        "ingestion_date": ingestion_date,

        "job_count": len(data),

        "jobs": data
    }


    # --------------------------------------------------------
    # Convert payload to JSON
    # --------------------------------------------------------

    json_data = json.dumps(
        output,
        ensure_ascii=False
    )


    # --------------------------------------------------------
    # Build S3 path
    # --------------------------------------------------------

    output_path = (
        f"{S3_RAW_PATH}/"
        f"ingestion_date={ingestion_date}/"
        f"{site}.json"
    )


    # --------------------------------------------------------
    # Write JSON to AWS S3
    # --------------------------------------------------------

    try:

        dbutils.fs.put(
            output_path,
            json_data,
            overwrite=True
        )

        print(
            "\nRAW data written successfully:"
            f"\n{output_path}"
        )


        successful_companies.append({

            "company": company_name,

            "jobs": len(data),

            "path": output_path

        })


    except Exception as e:

        print(
            f"ERROR writing "
            f"{company_name} to S3"
        )

        print(str(e))

        failed_companies.append(
            company_name
        )


# ============================================================
# 6. INGESTION SUMMARY
# ============================================================

print("\n")
print("==========================================")
print("LEVER INGESTION SUMMARY")
print("==========================================")

print(
    f"Successful companies: "
    f"{len(successful_companies)}"
)

print(
    f"Failed companies: "
    f"{len(failed_companies)}"
)


# ============================================================
# 7. SUCCESSFUL COMPANIES
# ============================================================

print("\nSuccessful:")

for item in successful_companies:

    print(
        f"{item['company']} → "
        f"{item['jobs']} jobs → "
        f"{item['path']}"
    )


# ============================================================
# 8. FAILED COMPANIES
# ============================================================

print("\nFailed:")

if failed_companies:

    for company in failed_companies:

        print(company)

else:

    print("None")


# ============================================================
# 9. FINAL STATUS
# ============================================================

print("\n==========================================")

if len(failed_companies) == 0:

    print(
        "LEVER INGESTION COMPLETED "
        "SUCCESSFULLY"
    )

else:

    print(
        "LEVER INGESTION COMPLETED "
        "WITH FAILURES"
    )

print("==========================================")

Ingestion date: 2026-10-06
Run timestamp: 2026-10-06T12:52:06.606003+00:00

------------------------------------------
Fetching: Palantir
Endpoint: https://api.lever.co/v0/postings/palantir
HTTP Status: 200
Jobs received: 316
Wrote 6126028 bytes.

RAW data written successfully:
s3://job-market-analysis-pipeline/raw-data/lever/ingestion_date=2026-10-06/palantir.json

------------------------------------------
Fetching: Spotify
Endpoint: https://api.lever.co/v0/postings/spotify
HTTP Status: 200
Jobs received: 76
Wrote 752980 bytes.

RAW data written successfully:
s3://job-market-analysis-pipeline/raw-data/lever/ingestion_date=2026-10-06/spotify.json

------------------------------------------
Fetching: Zoox
Endpoint: https://api.lever.co/v0/postings/zoox
HTTP Status: 200
Jobs received: 229
Wrote 2316877 bytes.

RAW data written successfully:
s3://job-market-analysis-pipeline/raw-data/lever/ingestion_date=2026-10-06/zoox.json

------------------------------------------
Fetching: Kraken
En